In [1]:
import pandas as pd
import requests
from secret.key import NAVER_API_CLIENT_SECRET, NAVER_API_CLIENT_ID, NAVER_MAPS_CLIENT_ID, NAVER_MAPS_CLIENT_SECRET

In [2]:
df = pd.read_excel(
    "1톤화물차 운송 내역.xlsx",
    sheet_name="납품지",
    header=0
)

In [3]:
df.head(3)

,납품업체,업체명,소재지,주소,거리_km
0,가양,가양산업,경기 용인,경기도 용인시 처인구 양지읍 주북로193번길 3 가양산업,64.809
1,고려,고려엔티씨,경기 안성,경기도 안성시 죽산면 용설로 607-52,26.492
2,광일,광일전선,경기 포천,경기도 포천시 가산면 시우동길 150-2,130.704


In [4]:
# Maps API
MAP_CLIENT_ID = NAVER_MAPS_CLIENT_ID
MAP_CLIENT_SECRET = NAVER_MAPS_CLIENT_SECRET

# NAVER API HUB
SEARCH_CLIENT_ID = NAVER_API_CLIENT_ID
SEARCH_CLIENT_SECRET = NAVER_API_CLIENT_SECRET

COMPANY_ADDRESS = "충청북도 음성군 맹동면 덕금로 234번길 89"

# 업체명 검색
def search_company(company_name, location):

    # 소재지가 없으면 업체명만 검색
    if pd.isna(location) or str(location).strip() == "":
        query = company_name
    else:
        query = f"{company_name} {location}"

    url = "https://naverapihub.apigw.ntruss.com/search/v1/local"

    headers = {
        "X-NCP-APIGW-API-KEY-ID": SEARCH_CLIENT_ID,
        "X-NCP-APIGW-API-KEY": SEARCH_CLIENT_SECRET,
    }

    res = requests.get(
        url,
        headers=headers,
        params={
            "query": query,
            "display": 1,
        },
    ).json()

    if not res["items"]:
        return None

    item = res["items"][0]

    address = item["roadAddress"] or item["address"]

    # 지역검색 결과에 좌표가 포함되어 있음
    # 지역검색 API의 좌표는 경위도에 10^7을 곱한 정수임
    x = float(item["mapx"]) / 10_000_000
    y = float(item["mapy"]) / 10_000_000

    return address, (x, y)


# 주소 → 좌표
def geocode(address):

    url = "https://maps.apigw.ntruss.com/map-geocode/v2/geocode"

    headers = {
        "x-ncp-apigw-api-key-id": MAP_CLIENT_ID,
        "x-ncp-apigw-api-key": MAP_CLIENT_SECRET,
    }

    res = requests.get(
        url,
        headers=headers,
        params={"query": address},
    ).json()

    data = res["addresses"][0]

    return float(data["x"]), float(data["y"])


# 자동차 거리
def get_distance(start, goal):

    url = "https://maps.apigw.ntruss.com/map-direction/v1/driving"

    headers = {
        "x-ncp-apigw-api-key-id": MAP_CLIENT_ID,
        "x-ncp-apigw-api-key": MAP_CLIENT_SECRET,
    }

    response = requests.get(
        url,
        headers=headers,
        params={
            "start": f"{start[0]},{start[1]}",
            "goal": f"{goal[0]},{goal[1]}",
            "option": "traoptimal",
        },
    )
    response.raise_for_status()
    res = response.json()

    if "route" not in res:
        raise RuntimeError(f"경로 API 오류: {res}")

    summary = res["route"]["traoptimal"][0]["summary"]

    return summary["distance"] / 1000


# 우리 회사 좌표
company_coord = geocode(COMPANY_ADDRESS)


addresses = []
distances = []


for _, row in df.iterrows():

    result = search_company(
        row["업체명"],
        row["소재지"],
    )

    if result is None:
        addresses.append(None)
        distances.append(None)
        continue

    address, vendor_coord = result

    distance = get_distance(
        company_coord,
        vendor_coord,
    )

    addresses.append(address)
    distances.append(distance)

df["주소"] = addresses
df["거리_km"] = distances

print('작업 완료')

In [7]:
df.to_excel("납품지.xlsx", index=False)